In [2]:
import pandas as pd

quality = pd.read_csv(r"\\NAS3_Z\all\BKP_PERE\BBDD_Datasets\TEST_ECHOAVC\inference_predictions_20250903_114612_calidad_1.csv")
view = pd.read_csv(r"\\NAS3_Z\all\BKP_PERE\BBDD_Datasets\TEST_ECHOAVC\probabilities_basal.csv")

In [3]:
quality['video'] = quality['path'].apply(lambda x: x.split('\\')[-1].split('.')[0])
quality['study'] = quality['path'].apply(lambda x: x.split('\\')[-2])
quality_df = quality[['study', 'video', 'calidad_pred']]
quality_df

,study,video,calidad_pred
0,1_001,US-0001-044,"[0.687426745891571, 0.1608392745256424, -0.971..."
1,1_001,US-0001-045,"[0.9688097238540649, 0.13805519044399261, -1.2..."
2,1_001,US-0001-046,"[1.0587979555130005, 0.09039976447820663, -1.3..."
3,1_001,US-0001-047,"[0.6897500157356262, 0.12478344887495041, -1.0..."
4,1_001,US-0001-034,"[1.2625453472137451, 0.6494037508964539, -1.67..."
5,1_001,US-0001-058,"[-1.38197660446167, 0.44491249322891235, 1.301..."
6,1_001,US-0001-060,"[-0.8807868957519531, 0.35776475071907043, 0.6..."
7,1_002,US-0001-086,"[-0.17293071746826172, 0.34299203753471375, 0...."
8,1_002,US-0001-087,"[-0.01918274722993374, 0.4005874693393707, -0...."
9,1_002,US-0001-088,"[0.09966246038675308, 0.33751046657562256, -0...."


In [4]:
quality_df.to_csv('EchoAVC/data/quality.csv', index=False)

In [5]:
# Función para extraer study y fname
def extract_study_and_fname(path):
    parts = path.split('\\')
    # Extraer fname (sin .dcm)
    fname = parts[-1].replace('.dcm', '')
    
    # substudy = parts[-2].split(' ')[0]

    study = parts[-2].split(' ')[0]

    return study, fname

# Aplicar la función
view[['study', 'fname']] = view['image'].apply(lambda x: pd.Series(extract_study_and_fname(x)))
# Agrupar columnas por vistas
view['A2C'] = view[['a2c_lvocc_s', 'a2c_laocc', 'a2c']].sum(axis=1)
view['A3C'] = view[['a3c_lvocc_s', 'a3c_laocc', 'a3c']].sum(axis=1)
view['A4C'] = view[['a4c_lvocc_s', 'a4c_laocc', 'a4c']].sum(axis=1)
view['A5C'] = view['a5c']
view['PLAX'] = view[['plax_far', 'plax_plax', 'plax_laz', 'plax_lac']].sum(axis=1)
view['PSAX_OTHER'] = view[['psax_pap', 'psax_apex']].sum(axis=1)
view['PSAX_MV'] = view[['psax_mv', 'psax_pap', 'psax_apex']].sum(axis=1)
view['PSAX_AO'] = view[['psax_az', 'psax_avz']].sum(axis=1)
view['RV_Inflow'] = view['rvinf']
view['Subcostal'] = view['subcostal']

# Conservar 'other' y 'suprasternal'
df_views = view[['study', 'fname', 'A2C', 'A3C', 'A4C', 'A5C', 'PLAX', 'PSAX_MV', 'PSAX_AO', 'PSAX_OTHER', 'RV_Inflow', 'Subcostal', 'other', 'suprasternal']]
view_columns = [
    'A2C', 'A3C', 'A4C', 'A5C', 'PLAX', 
    'PSAX_MV', 'PSAX_AO', 'PSAX_OTHER', 'RV_Inflow', 'Subcostal',
    'other', 'suprasternal'
]

# Crear una nueva columna 'view' con el nombre de la columna con el valor máximo
df_views['view'] = df_views[view_columns].idxmax(axis=1)

# añade una columna que basicamente sea view_prob, donde mire que valor hay en view y coja ese valor de la columna correspondiente
def get_view_prob(row):
    view = row['view']
    return row[view]
df_views['view_prob'] = df_views.apply(get_view_prob, axis=1)
df_views_sel = df_views[['study', 'fname', 'view', 'view_prob']]
df_views_sel.rename(columns={'fname': 'video'}, inplace=True)
df_views_sel

e:\25366074H\AppData\Local\Temp\ipykernel_20444\2688359766.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_views['view'] = df_views[view_columns].idxmax(axis=1)
e:\25366074H\AppData\Local\Temp\ipykernel_20444\2688359766.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_views['view_prob'] = df_views.apply(get_view_prob, axis=1)
e:\25366074H\AppData\Local\Temp\ipykernel_20444\2688359766.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the cav

,study,video,view,view_prob
0,1_001,US-0001-034,PLAX,0.9997
1,1_001,US-0001-035,PSAX_AO,0.7139
2,1_001,US-0001-036,PSAX_AO,0.7858
3,1_001,US-0001-037,PSAX_AO,0.7807
4,1_001,US-0001-039,PSAX_AO,0.9950
...,...,...,...,...
84,1_003,US-0001-233,A3C,0.4948
85,1_003,US-0001-237,PSAX_AO,0.9918
86,1_003,US-0001-238,other,0.9998
87,1_003,US-0001-241,suprasternal,0.3688


In [6]:
df_views_sel.to_csv('EchoAVC/data/view.csv', index=False)